# 12 - IVFPQ (Inverted File with Product Quantization)


---

In the previous notebooks, we learned:

- IVF reduces the search space.
- Product Quantization (PQ) reduces memory usage.

Now we combine these two powerful ideas into one index:

**IVFPQ**.

IVFPQ is one of the most widely used indexing techniques for billion-scale vector search.

## History

Researchers faced two major challenges.

Problem 1

Searching every vector was slow.

↓

IVF solved this by searching only a few clusters.

Problem 2

Storing millions of vectors required a lot of memory.

↓

PQ solved this by compressing vectors.

Researchers then combined both ideas.

The result was **IVFPQ**.

## Think Like a Researcher

Imagine a huge library.

Instead of searching every shelf,

you first go to the correct section.

Then,

instead of storing every book in full,

you store a compact summary.

IVFPQ follows the same idea.

First,

find the right cluster.

Then,

search compressed vectors inside that cluster.

## IVFPQ Pipeline

```
Documents

↓

Embedding Model

↓

Vectors

↓

KMeans Clustering (IVF)

↓

Clusters

↓

Compress Vectors (PQ)

↓

Store Compressed Codes

↓

User Query

↓

Nearest Cluster

↓

Search Compressed Vectors

↓

Top K Results
```

In [1]:
clusters = {
    0:["Doc A","Doc B"],
    1:["Doc C","Doc D"],
    2:["Doc E","Doc F"]
}

query_cluster = 1

clusters[query_cluster]

['Doc C', 'Doc D']

## Observation

Instead of searching every document,

IVFPQ searches only

Cluster 1.

This reduces the search space.

Inside the cluster,

vectors are stored in compressed form,

saving memory.

In [2]:
total_vectors = 1_000_000

cluster_vectors = 20_000

print("Total vectors :", total_vectors)

print("Vectors searched :", cluster_vectors)

Total vectors : 1000000
Vectors searched : 20000


In [3]:
dimension = 768

float32_size = 4

original = dimension * float32_size

compressed = 64

print("Original :", original, "bytes")

print("Compressed :", compressed, "bytes")

Original : 3072 bytes
Compressed : 64 bytes


## Why is IVFPQ Faster?

Two optimizations happen together.

1.

IVF

↓

Search fewer vectors.

2.

PQ

↓

Smaller vectors.

Together,

IVFPQ reduces both

search time

and

memory usage.

In [4]:
import faiss

dimension = 128

nlist = 100

m = 8

bits = 8

quantizer = faiss.IndexFlatL2(dimension)

index = faiss.IndexIVFPQ(
    quantizer,
    dimension,
    nlist,
    m,
    bits
)

print(index)

<faiss.swigfaiss.IndexIVFPQ; proxy of <Swig Object of type 'faiss::IndexIVFPQ *' at 0x00000217C580CB70> >


## Parameter Explanation

dimension

↓

Embedding dimension

nlist

↓

Number of IVF clusters

m

↓

Number of subvectors

bits

↓

Bits used to encode each subvector

In [5]:
import numpy as np

np.random.seed(42)

train_vectors = np.random.random((5000,128)).astype("float32")

index.train(train_vectors)

print(index.is_trained)

True


In [6]:
database = np.random.random((10000,128)).astype("float32")

index.add(database)

print(index.ntotal)

10000


In [7]:
query = np.random.random((1,128)).astype("float32")

distance, ids = index.search(query,5)

print(ids)

print(distance)

[[8776 1818 2835 5042  391]]
[[12.48741  12.505743 12.516115 12.630958 12.79355 ]]


## Internal Working

```
Query

↓

Nearest IVF Cluster

↓

Compressed Vectors

↓

Approximate Distance

↓

Top K
```

## Advantages

✅ Faster than Flat Index

✅ Lower memory usage

✅ Good for very large datasets

✅ Used in FAISS

✅ Suitable for billion-scale search

## Limitations

❌ Approximate search

❌ Requires training

❌ More parameters to tune

❌ Compression may slightly reduce accuracy

## Comparison

| Method | Search Space | Memory | Accuracy |
|---------|--------------|--------|----------|
| Flat | All vectors | High | Highest |
| IVF | Few clusters | High | High |
| PQ | All vectors | Low | High |
| IVFPQ | Few clusters | Low | High |

##  Applications

IVFPQ is commonly used in

- Large RAG systems
- Image retrieval
- Video search
- Recommendation systems
- Billion-scale semantic search

## Summary

Today I learned:

- What IVFPQ is
- Why it combines IVF and PQ
- How it reduces search time
- How it saves memory
- FAISS IVFPQ basics

##  Think Like a Researcher

IVFPQ is fast and memory efficient.

But another question appears.

Can we search without trees,

without clusters,

and without compression?

What if vectors themselves become connected like a graph,

and we navigate through that graph to find nearest neighbors?

This idea became **HNSW (Hierarchical Navigable Small World)**.